# Notebook 01 — Ingestion Exploration
Use this notebook to inspect parsed documents, verify table extraction, and debug the ingestion pipeline before running at full scale.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

from pathlib import Path
from config import SEC_DATA_PATH, CHUNK_SIZE, CHUNK_OVERLAP

print('SEC_DATA_PATH:', SEC_DATA_PATH)
print('CHUNK_SIZE   :', CHUNK_SIZE)
print('CHUNK_OVERLAP:', CHUNK_OVERLAP)

## 1. Inspect raw file count by company

In [ ]:
from collections import Counter

files = list(SEC_DATA_PATH.rglob('*'))
files = [f for f in files if f.is_file() and f.suffix.lower() in {'.html', '.htm', '.pdf'}]

# Count by ticker (first directory level below SEC_DATA_PATH)
ticker_counts = Counter()
for f in files:
    parts = f.relative_to(SEC_DATA_PATH).parts
    if parts:
        ticker_counts[parts[0]] += 1

print(f'Total files : {len(files)}')
print(f'\nFiles by ticker:')
for ticker, count in sorted(ticker_counts.items()):
    print(f'  {ticker:10s} {count}')

## 2. Parse a single file and inspect elements

In [ ]:
from ingestion import _partition_file

# Change this to any file in your dataset
sample_file = files[0]
print('Parsing:', sample_file)

elements = _partition_file(sample_file)
print(f'Total elements: {len(elements)}')

# Category breakdown
from collections import Counter
categories = Counter(getattr(e, 'category', 'Unknown') for e in elements)
print('\nElement categories:')
for cat, count in categories.most_common():
    print(f'  {cat:30s} {count}')

## 3. Inspect table elements

In [ ]:
from markdownify import markdownify as md
from IPython.display import Markdown, display

table_elements = [e for e in elements if getattr(e, 'category', '') == 'Table']
print(f'Table elements found: {len(table_elements)}')

# Display first 3 tables as Markdown
for i, tbl in enumerate(table_elements[:3]):
    raw_html = getattr(tbl.metadata, 'text_as_html', None)
    if raw_html:
        table_md = md(raw_html)
        display(Markdown(f'### Table {i+1}\n\n{table_md}'))
    else:
        print(f'Table {i+1} (no HTML):', str(tbl)[:300])

## 4. Run full ingestion on a single company folder

In [ ]:
import ingestion
from unittest.mock import patch

# Override SEC_DATA_PATH to a single company for quick testing
SINGLE_COMPANY = SEC_DATA_PATH / 'AAPL'   # change ticker as needed

with patch.object(ingestion, 'SEC_DATA_PATH', SINGLE_COMPANY):
    with patch.object(ingestion, 'CHECKPOINT_FILE', ingestion.Path('./test_checkpoint.json')):
        docs = ingestion.ingest_filings()

print(f'\nDocuments produced: {len(docs)}')

tables = [d for d in docs if d.metadata['chunk_type'] == 'table']
texts  = [d for d in docs if d.metadata['chunk_type'] == 'text']
print(f'  Table chunks : {len(tables)}')
print(f'  Text chunks  : {len(texts)}')

## 5. Inspect sample documents

In [ ]:
import pandas as pd

meta_df = pd.DataFrame([d.metadata for d in docs])
display(meta_df.head(20))

# Show longest table chunk
if tables:
    longest = max(tables, key=lambda d: len(d.page_content))
    from IPython.display import Markdown, display
    display(Markdown('### Longest Table Chunk\n\n' + longest.page_content))

## 6. Chunk size distribution

In [ ]:
import plotly.express as px

lengths = [len(d.page_content) for d in docs]
chunk_types = [d.metadata['chunk_type'] for d in docs]

size_df = pd.DataFrame({'length': lengths, 'chunk_type': chunk_types})
fig = px.histogram(
    size_df, x='length', color='chunk_type', nbins=50,
    title='Chunk Size Distribution',
    labels={'length': 'Characters', 'chunk_type': 'Type'},
    barmode='overlay', opacity=0.7
)
fig.show()